# Notebook 05 — Descriptor-enhanced CYP inhibition baselines

**Goal:** compare ECFP4, count-Morgan + RDKit descriptors, and ECFP4 + RDKit + selected chemistry features using **identical five-fold scaffold-grouped splits** for all four CYP endpoints. Report out-of-fold (OOF) predictions and save a **reusable compound-level fold manifest** for Notebook 06.

This is a direct-inhibition pIC50 project, not a TDI classifier. **No challenge test labels are used.** Only compounds with available pIC50 labels are scored; missing targets remain missing.

**Validation:** Entire Bemis–Murcko scaffold groups are held out; acyclic molecules are grouped by canonical SMILES. Repeated scaffolds never cross folds. Feature calculations do not use experimental endpoints. Missing-value imputers are fit on training folds only. The selected hyperparameters are deliberately fixed rather than tuned on OOF outcomes.

**Scoring caution:** We report MAE/R²/Spearman/Kendall plus *mean interval-excess distance* (zero within experimental confidence limits), **not the organizers' official MA-ST-RAE**. Official ST-RAE additionally uses a relative normalization; do not compare these figures numerically with leaderboard scores without reproducing that implementation.

**Usage:** Runtime → Run all in Google Colab. To test offline, set `DATA_OVERRIDE` to a local CSV path below. Notebook 06 uses the exported `fold_manifest.csv` and `oof_predictions.csv`.

## 0 · Install packages (Google Colab, CPU)

In [ ]:
%pip -q install "rdkit>=2023.9" "scikit-learn>=1.6" "lightgbm>=4.3" "scipy>=1.11" pandas matplotlib seaborn

## 1 · Configuration, provenance, and data loading

In [ ]:
from pathlib import Path
import os, io, json, sys, platform, hashlib, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, Crippen, Lipinski
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import spearmanr, kendalltau

SEED=2026
N_SPLITS=5
FOLDS_TO_RUN=None  # None means all 5; set [0] for a short smoke test (not complete OOF).
FAST_MODE=False    # True: 50 trees, only the first 2 variants, for testing the workflow.
DATA_OVERRIDE=None # e.g. '/content/synthetic_cyp.csv' for testing without network
DATA_URL='https://huggingface.co/datasets/openadmet/cyp-challenge-train-test/resolve/main/cyp-challenge-TRAIN_inhibition.csv'
ISOFORMS=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
OUT=Path('cyp_notebook_05_results');OUT.mkdir(exist_ok=True)
np.random.seed(SEED)
if DATA_OVERRIDE:
    data_bytes=Path(DATA_OVERRIDE).read_bytes()
else:
    import urllib.request
    with urllib.request.urlopen(DATA_URL, timeout=120) as response: data_bytes=response.read()
source_sha256=hashlib.sha256(data_bytes).hexdigest()
df=pd.read_csv(io.BytesIO(data_bytes))
print('Input shape:',df.shape,'SHA256:',source_sha256)
print('Columns:',df.columns.to_list())
print('RDKit',rdBase.rdkitVersion,'LightGBM',lgb.__version__)
assert 'SMILES' in df.columns, 'Expected SMILES column not found; inspect dataset schema.'
for iso in ISOFORMS:
    assert f'{iso}_pIC50_direct_inhibition' in df.columns, f'Missing target for {iso}'

## 2 · Canonicalize compounds, check uniqueness, and create a five-fold manifest

In [ ]:
rows=[]; bad=[]
for src_row,s in enumerate(df['SMILES']):
    mol=Chem.MolFromSmiles(s) if isinstance(s,str) else None
    if mol is None:
        bad.append(src_row);continue
    canon=Chem.MolToSmiles(mol,canonical=True,isomericSmiles=True)
    core=MurckoScaffold.GetScaffoldForMol(mol)
    scaffold=Chem.MolToSmiles(core, canonical=True, isomericSmiles=False) if core.GetNumAtoms() else 'ACYCLIC_'+canon
    rows.append((src_row,canon,scaffold,mol))
assert rows, 'No parseable molecules.'
clean=df.iloc[[r[0] for r in rows]].copy().reset_index(drop=True)
clean['canonical_smiles']=[r[1] for r in rows]
clean['scaffold_group']=[r[2] for r in rows]
if clean['canonical_smiles'].duplicated().any():
    raise ValueError('Duplicate canonical structures found. Aggregate labels deliberately before fold assignment.')
assert clean['scaffold_group'].nunique() >= N_SPLITS
mols=[r[3] for r in rows]
folds=np.full(len(clean),-1,dtype=int)
# GroupKFold's deterministic greedy balancing keeps related scaffolds in one fold.
for fold,(_,vi) in enumerate(GroupKFold(n_splits=N_SPLITS).split(clean, groups=clean['scaffold_group'])):
    folds[vi]=fold
assert (folds>=0).all()
assert all(len(set(clean.loc[folds!=f,'scaffold_group']) & set(clean.loc[folds==f,'scaffold_group']))==0 for f in range(N_SPLITS))
clean['fold']=folds
manifest=clean[['canonical_smiles','scaffold_group','fold']].copy()
if 'Molecule_Name' in clean:manifest.insert(0,'Molecule_Name',clean['Molecule_Name'].values)
manifest.to_csv(OUT/'fold_manifest.csv',index=False)
print('Valid structures:',len(clean),'rejected:',len(bad),'scaffold groups:',manifest.scaffold_group.nunique())
print(manifest['fold'].value_counts().sort_index())
display(pd.DataFrame({'isoform':ISOFORMS,'n_labeled':[int(pd.to_numeric(clean[f'{i}_pIC50_direct_inhibition'],errors='coerce').notna().sum()) for i in ISOFORMS]}))

## 3 · Compute molecular fingerprints and explicit descriptors once

In [ ]:
from rdkit.Chem import rdFingerprintGenerator
n_bits=2048
binary_gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=n_bits)
count_gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=n_bits,countSimulation=False)
X_binary=np.zeros((len(clean),n_bits),np.float32)
X_count=np.zeros((len(clean),n_bits),np.float32)
for ix,mol in enumerate(mols):
    X_binary[ix]=np.asarray(binary_gen.GetFingerprintAsNumPy(mol),dtype=np.float32)
    X_count[ix]=np.asarray(count_gen.GetCountFingerprintAsNumPy(mol),dtype=np.float32)
# Most RDKit 2D descriptors, including charge/VSA/BCUT when available.
desc_names=[name for name,func in Descriptors._descList]
desc_functions=[func for name,func in Descriptors._descList]
D=np.empty((len(mols),len(desc_names)),dtype=np.float32)
for i,mol in enumerate(mols):
    for j,func in enumerate(desc_functions):
        try: D[i,j]=float(func(mol))
        except Exception: D[i,j]=np.nan
D[~np.isfinite(D)]=np.nan
# Explicit chemistry-related structural counts; do not claim these are predicted pKa values.
smarts={
 'basic_amine_count':'[NX3;H0,H1,H2;!$(NC=O)]',
 'aromatic_n_count':'[n]',
 'carboxylate_acid_count':'[CX3](=O)[OX1H0-,OX2H1]',
 'phenol_count':'[OX2H][c]',
 'halogen_count':'[F,Cl,Br,I]',
 'sulfonamide_count':'[SX4](=O)(=O)[NX3]',
 'tertiary_amine_count':'[NX3;H0;!$(NC=O)]',
}
queries={name:Chem.MolFromSmarts(sm) for name,sm in smarts.items()}
assert all(q is not None for q in queries.values())
extra=np.array([[len(mol.GetSubstructMatches(q,uniquify=True)) for q in queries.values()] for mol in mols],dtype=np.float32)
feature_names={
 'ECFP4_binary':[f'ecfp4_bit_{k}' for k in range(n_bits)],
 'Morgan_count_RDKit':[f'count_{k}' for k in range(n_bits)]+desc_names,
 'ECFP4_RDKit_chemistry':[f'ecfp4_bit_{k}' for k in range(n_bits)]+desc_names+list(smarts),
}
features={
 'ECFP4_binary':X_binary,
 'Morgan_count_RDKit':np.hstack([X_count,D]),
 'ECFP4_RDKit_chemistry':np.hstack([X_binary,D,extra]),
}
for name,x in features.items():print(name,x.shape,'missing:',int(np.isnan(x).sum()))
assert np.isfinite(X_binary).all() and np.isfinite(X_count).all()
# Features are structural only, so their direct calculation across structures is not label leakage.
np.savez_compressed(OUT/'structural_features.npz', ECFP4_binary=X_binary, Morgan_count=X_count, RDKit_2D=D, Chemistry_counts=extra)
pd.Series(desc_names,name='descriptor').to_csv(OUT/'descriptor_names.csv',index=False)

## 4 · Train the same LightGBM model on each representation and each outer fold

In [ ]:
models_to_run=list(features)[:2] if FAST_MODE else list(features)
selected_folds=list(range(N_SPLITS)) if FOLDS_TO_RUN is None else list(FOLDS_TO_RUN)
assert set(selected_folds).issubset(set(range(N_SPLITS)))
records=[];metrics=[]
for feat_name in models_to_run:
    X=features[feat_name]
    for fold in selected_folds:
        vi=np.flatnonzero(folds==fold)
        ti=np.flatnonzero(folds!=fold)
        assert not (set(ti)&set(vi))
        imputer=SimpleImputer(strategy='median',add_indicator=False,keep_empty_features=True)
        Xtr=imputer.fit_transform(X[ti]); Xva=imputer.transform(X[vi])
        # NaN imputation is learned exclusively on the outer training fold.
        for iso in ISOFORMS:
            col=f'{iso}_pIC50_direct_inhibition'
            y=pd.to_numeric(clean[col],errors='coerce').to_numpy(float)
            train_idx=np.flatnonzero((folds!=fold)&np.isfinite(y))
            valid_idx=np.flatnonzero((folds==fold)&np.isfinite(y))
            if len(train_idx)<20 or len(valid_idx)<3:
                print('SKIP insufficient labels',iso,fold);continue
            # Xtr is indexed by ti and Xva is indexed by vi; map original row indices to fold-local indices.
            trpos=np.searchsorted(ti,train_idx)
            vpos=np.searchsorted(vi,valid_idx)
            reg=lgb.LGBMRegressor(
                n_estimators=50 if FAST_MODE else 350,
                learning_rate=0.04,
                num_leaves=15,
                max_depth=-1,
                min_child_samples=15,
                colsample_bytree=0.8,
                subsample=0.85,
                subsample_freq=1,
                reg_lambda=3.0,
                verbosity=-1,
                n_jobs=2,
                random_state=SEED,
            )
            reg.fit(Xtr[trpos],y[train_idx])
            yp=reg.predict(Xva[vpos]); yt=y[valid_idx]
            lo_col=f'{iso}_pIC50_direct_inhibition_conf_low'; hi_col=f'{iso}_pIC50_direct_inhibition_conf_high'
            lo=pd.to_numeric(clean.iloc[valid_idx][lo_col],errors='coerce').to_numpy(float) if lo_col in clean else np.full(len(yp),np.nan)
            hi=pd.to_numeric(clean.iloc[valid_idx][hi_col],errors='coerce').to_numpy(float) if hi_col in clean else np.full(len(yp),np.nan)
            interval_excess=np.maximum(lo-yp,0)+np.maximum(yp-hi,0)
            interval_excess[~np.isfinite(lo)|~np.isfinite(hi)|(lo>hi)]=np.nan
            rho=spearmanr(yt,yp).statistic if len(yp)>2 else np.nan
            tau=kendalltau(yt,yp).statistic if len(yp)>2 else np.nan
            metrics.append(dict(model=feat_name,fold=fold,isoform=iso,n_train=len(train_idx),n_valid=len(valid_idx),
                mae=mean_absolute_error(yt,yp),r2=r2_score(yt,yp),spearman=rho,kendall=tau,
                mean_interval_excess=float(np.nanmean(interval_excess)) if np.isfinite(interval_excess).any() else np.nan))
            for idx, truth, pred, low, high in zip(valid_idx,yt,yp,lo,hi):
                records.append(dict(canonical_smiles=clean.loc[idx,'canonical_smiles'],row_index=int(idx),fold=int(fold),
                    isoform=iso,model=feat_name,y_true=float(truth),y_pred=float(pred),ci_low=float(low),ci_high=float(high)))
            print(f'{feat_name:23s} fold={fold} {iso:7s} n={len(yp):4d} MAE={mean_absolute_error(yt,yp):.3f}')
preds=pd.DataFrame(records); metric_df=pd.DataFrame(metrics)
assert not preds.duplicated(['canonical_smiles','fold','isoform','model']).any()
preds.to_csv(OUT/'oof_predictions.csv',index=False)
metric_df.to_csv(OUT/'fold_metrics.csv',index=False)
display(metric_df.head(12))

## 5 · Compare OOF metrics and quantify predicted-range compression

In [ ]:
if len(preds):
    summary=[]
    for (model,iso),part in preds.groupby(['model','isoform']):
        a=part.y_true.to_numpy();b=part.y_pred.to_numpy()
        lo=part.ci_low.to_numpy();hi=part.ci_high.to_numpy()
        interval=np.maximum(lo-b,0)+np.maximum(b-hi,0)
        interval[~np.isfinite(lo)|~np.isfinite(hi)|(lo>hi)]=np.nan
        summary.append(dict(model=model,isoform=iso,n_oof=len(part),mae=mean_absolute_error(a,b),r2=r2_score(a,b),
            spearman=spearmanr(a,b).statistic,kendall=kendalltau(a,b).statistic,
            pred_sd_over_true_sd=float(np.std(b)/np.std(a)),
            mean_interval_excess=float(np.nanmean(interval)) if np.isfinite(interval).any() else np.nan))
    summary=pd.DataFrame(summary)
    summary.to_csv(OUT/'oof_summary.csv',index=False)
    display(summary.round(3))
    fig,ax=plt.subplots(figsize=(10,5))
    for model,group in summary.groupby('model'):
        ax.plot(group.isoform,group.mae,marker='o',label=model)
    ax.set(ylabel='OOF MAE (pIC50)',title='Direct-inhibition molecular feature comparison');ax.legend(fontsize=8)
    plt.tight_layout();plt.savefig(OUT/'01_oof_mae_comparison.png',dpi=160);plt.show()
    fig,ax=plt.subplots(figsize=(10,4))
    for model,group in summary.groupby('model'):
        ax.plot(group.isoform,group.pred_sd_over_true_sd,marker='o',label=model)
    ax.axhline(1,linestyle='--',color='gray',linewidth=1)
    ax.set(ylabel='Predicted SD / Experimental SD',title='Regression-to-mean diagnostic')
    ax.legend(fontsize=8);plt.tight_layout();plt.savefig(OUT/'02_predicted_range_compression.png',dpi=160);plt.show()

## 6 · Save reproducibility metadata and package results

In [ ]:
meta={
  'notebook':'05_descriptor_enhanced_baselines.ipynb','seed':SEED,'n_splits':N_SPLITS,
  'selected_folds':selected_folds,'complete_oof':len(selected_folds)==N_SPLITS,
  'models':models_to_run,'dataset_url':DATA_URL if DATA_OVERRIDE is None else DATA_OVERRIDE,
  'dataset_sha256':source_sha256,'n_molecules':int(len(clean)),'n_invalid':int(len(bad)),
  'fingerprint_radius':2,'n_bits':n_bits,'descriptor_count':len(desc_names),
  'lightgbm_version':lgb.__version__,'rdkit_version':rdBase.rdkitVersion,
  'python_version':platform.python_version(),
  'metric_note':'mean_interval_excess is not the official MA-ST-RAE scoring function',
  'fold_note':'GroupKFold on Murcko scaffolds, acyclic compounds use canonical isomeric SMILES',
}
(OUT/'metadata.json').write_text(json.dumps(meta,indent=2))
archive=Path('cyp_notebook_05_results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file():z.write(f,f'{OUT.name}/{f.name}')
print('Saved:',archive,archive.stat().st_size/1024,'KiB')
print('Upload this ZIP to Notebook 06 to reuse the EXACT fold manifest.')